In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import optuna
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, r2_score

from scipy.sparse.linalg import svds
from scipy.sparse import coo_matrix

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/movies.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/ratings.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/README.txt
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/tags.csv
/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/links.csv


<div style="
  border-radius: 20px;
  padding: 25px;
  background: radial-gradient(circle at top, #qf1323, #100043);
  border: 2px solid #00ffc8;
  text-align: center;
  box-shadow: 0 0 25px rgba(0, 255, 250, 0.25);
">
  <h1 style="
    font-size: 28px;
    font-family: 'Trebuchet MS', sans-serif;
    letter-spacing: 2px;
    color: #003ac8;
    text-shadow: 0 0 12px rgba(0,115,400,0.5);
  ">
    <span style="color:#7aa6ec5;">Load the DataFrame</span> 
  </h1>
</div>

In [5]:
df = pd.read_csv('/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/ratings.csv')
movies = pd.read_csv('/kaggle/input/datasets/shubhammehta21/movie-lens-small-latest-dataset/movies.csv') 

In [3]:
# Show first five rows
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [9]:
user_ids = df['userId'].unique().tolist()
user_map = {x: i for i, x in enumerate(user_ids)}

movie_ids = df['movieId'].unique().tolist()
movie_map = {x: i for i, x in enumerate(movie_ids)}

movie_ids2 = movies['movieId'].unique().tolist()
movie_map2 = {x: i for i, x in enumerate(movie_ids2)}

df['user_id'] = df['userId'].map(user_map)
df['movie_id'] = df['movieId'].map(movie_map)
movies['movie_id'] = movies['movieId'].map(movie_map2)

In [ ]:
df.describe()

<div style="
  border-radius: 20px;
  padding: 25px;
  background: radial-gradient(circle at top, #qf1323, #100043);
  border: 2px solid #00ffc8;
  text-align: center;
  box-shadow: 0 0 25px rgba(0, 255, 250, 0.25);
">
  <h1 style="
    font-size: 28px;
    font-family: 'Trebuchet MS', sans-serif;
    letter-spacing: 2px;
    color: #003ac8;
    text-shadow: 0 0 12px rgba(0,115,400,0.5);
  ">
    <span style="color:#7aa6ec5;">Splitting the Data into Train and Test</span> 
  </h1>
</div>

#### Added global number since the model was issuing <code>IndexError</code>: index 609 is out of bounds for axis 0 with size 609. So global number makes sure that the index is not out of bounds.

In [10]:
# Compute the global sizes before splitting
n_users_global = df['user_id'].max() + 1
n_movies_global = df['movie_id'].max() + 1
print(f"Global users: {n_users_global} | Global movies: {n_movies_global}")

Global users: 610 | Global movies: 9724


# Time Split

In [11]:
df = df.sort_values('timestamp') # Sorting the data

sort_index = int(len(df) * 0.64)

train_df = df[:sort_index]
test_df = df[sort_index:]

# Our model expect arrays 
train_users = train_df['user_id'].values
train_movies = train_df['movie_id'].values
train_ratings = train_df['rating'].values
train_timestamps = train_df['timestamp'].values

test_users = test_df['user_id'].values
test_movies = test_df['movie_id'].values
test_ratings = test_df['rating'].values
test_timestamps = test_df['timestamp'].values

print(f"Time-based split → Train: {len(train_df)} ratings (older) | Test: {len(test_df)} ratings (newer)")

Time-based split → Train: 64535 ratings (older) | Test: 36301 ratings (newer)


<div style="
  border-radius: 20px;
  padding: 25px;
  background: radial-gradient(circle at top, #qf1323, #100043);
  border: 2px solid #00ffc8;
  text-align: center;
  box-shadow: 0 0 25px rgba(0, 255, 250, 0.25);
">
  <h1 style="
    font-size: 28px;
    font-family: 'Trebuchet MS', sans-serif;
    letter-spacing: 2px;
    color: #003ac8;
    text-shadow: 0 0 12px rgba(0,115,400,0.5);
  ">
    <span style="color:#7aa6ec5;">MatrixFactorizationSGD with user and item biases, and temporal dynamics
    </span> 
  </h1>
</div>

In [32]:
class MatrixFactorizationSGD:
    """
    Matrix Factorization with Biases + Temporal Dynamics
    (User drift over time — the Netflix Prize upgrade!)
    """
    def __init__(self, n_factors=20, learning_rate=0.01, epochs=50, 
                 batch_size=1, reg=None, reg_param=0.01,
                 n_users=None, n_items=None):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.reg = reg
        self.reg_param = reg_param
        self.n_users = n_users
        self.n_items = n_items
        
        self.P = None
        self.Q = None
        self.global_bias = 0.0
        self.user_bias = None
        self.item_bias = None
        
        self.user_drift = None  
        self.item_drift = None

        self.t_min = None
        self.t_max = None

        self.seen_users = None
        self.seen_items = None

    def fit(self, user_ids, item_ids, ratings, timestamps):
        """timestamps must be provided (same length as ratings)"""
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        ratings = np.asarray(ratings, dtype=float)
        timestamps = np.asarray(timestamps, dtype=float)
        
        n_users = self.n_users if self.n_users is not None else (user_ids.max() + 1)
        n_items = self.n_items if self.n_items is not None else (item_ids.max() + 1)
        
        self.P = np.random.normal(0, 0.01, (n_users, self.n_factors))
        self.Q = np.random.normal(0, 0.01, (n_items, self.n_factors))
        self.global_bias = np.mean(ratings)
        
        self.user_bias = np.zeros(n_users)
        self.item_bias = np.zeros(n_items)

        self.user_drift = np.zeros(n_users)
        self.item_drift = np.zeros(n_items)
        
        # Normalize timestamp from 0 to 1 to prevent overfitting
        t_min, t_max = timestamps.min(), timestamps.max()
        self.t_min = t_min
        self.t_max = t_max
        t_norm = (timestamps - t_min) / (t_max - t_min) if t_max > t_min else np.zeros_like(timestamps)
        
        for epoch in range(self.epochs):
            indices = np.random.permutation(len(ratings))
            for i in range(0, len(ratings), self.batch_size):
                batch_idx = indices[i:i + self.batch_size]
                u_batch = user_ids[batch_idx]
                i_batch = item_ids[batch_idx]
                r_batch = ratings[batch_idx]
                t_batch = t_norm[batch_idx]
                
                pred = (np.sum(self.P[u_batch] * self.Q[i_batch], axis=1) 
                        + self.global_bias 
                        + self.user_bias[u_batch] 
                        + self.item_bias[i_batch]
                        + self.user_drift[u_batch] * t_batch  # multipy user bias with normalized timestamp
                        + self.item_drift[i_batch] * t_batch)
                
                error = r_batch - pred
                
                # Gradients (latent + biases unchanged)
                grad_P = -2 * (error[:, np.newaxis] * self.Q[i_batch])
                grad_Q = -2 * (error[:, np.newaxis] * self.P[u_batch])
                grad_user_bias = -2 * error
                grad_item_bias = -2 * error
                
                # TEMPORAL DYNAMICS: Gradient for user and item drift
                grad_user_drift = -2 * error * t_batch
                grad_item_drift = -2 * error * t_batch
                
                # Regularization (L2 on everything)
                if self.reg == 'l2':
                    grad_P += 2 * self.reg_param * self.P[u_batch]
                    grad_Q += 2 * self.reg_param * self.Q[i_batch]
                    grad_user_bias += 2 * self.reg_param * self.user_bias[u_batch]
                    grad_item_bias += 2 * self.reg_param * self.item_bias[i_batch]
                    grad_user_drift += 2 * self.reg_param * self.user_drift[u_batch]
                    grad_item_drift += 2 * self.reg_param * self.item_drift[i_batch]
                
                self.P[u_batch] -= self.learning_rate * grad_P
                self.Q[i_batch] -= self.learning_rate * grad_Q
                self.user_bias[u_batch] -= self.learning_rate * grad_user_bias
                self.item_bias[i_batch] -= self.learning_rate * grad_item_bias
                self.user_drift[u_batch] -= self.learning_rate * grad_user_drift   
                self.item_drift[i_batch] -= self.learning_rate * grad_item_drift 

        self.seen_users = set(user_ids)
        self.seen_items = set(item_ids)
            
        
        return self
    
    def predict(self, user_ids, item_ids, timestamps):
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        timestamps = np.asarray(timestamps, dtype=float)

        if self.t_min is None or self.t_max is None:
            raise ValueError("Call fit() first — temporal scale not stored.")
        t_norm = (timestamps - self.t_min) / (self.t_max - self.t_min)
        t_norm = np.clip(t_norm, 0.0, None)
        
        return (np.sum(self.P[user_ids] * self.Q[item_ids], axis=1) 
                + self.global_bias 
                + self.user_bias[user_ids] 
                + self.item_bias[item_ids]
                + self.user_drift[user_ids] * t_norm
                + self.item_drift[item_ids] * t_norm)

    def get_top_n_recommendations(self, user_id: int, top_n: int = 10, 
                              timestamp: float = None) -> tuple:
    
        if self.P is None:
            raise ValueError("Model must be fit first.")
        if user_id < 0 or user_id >= self.n_users:
            raise ValueError(f"user_id {user_id} out of range.")
    
        # All possible items (dense indices 0..n_items-1)
        all_items = np.arange(self.n_items)
    
        # Normalize time (use stored training scale)
        if timestamp is None:
            t_norm = np.full(len(all_items), 1.0)      
        else:
            t_norm = (np.full(len(all_items), timestamp) - self.t_min) / (self.t_max - self.t_min)
            t_norm = np.clip(t_norm, 0.0, None)
    
        # Cold-start check
        is_cold_start = (self.seen_users is not None and user_id not in self.seen_users)
    
        if is_cold_start:
            # Mean-user fallback (first-principles: we know nothing about this user)
            user_factor = np.zeros(self.n_factors)
            user_bias = 0.0
            user_drift = 0.0
        else:
            user_factor = self.P[user_id]
            user_bias = self.user_bias[user_id]
            user_drift = self.user_drift[user_id]
    
        # Vectorized prediction for all items
        dot_prod = np.dot(user_factor, self.Q[all_items].T)   # shape (n_items,)
        pred = (self.global_bias 
                + user_bias 
                + self.item_bias[all_items]
                + dot_prod
                + user_drift * t_norm
                + self.item_drift[all_items] * t_norm)
    
        # Top-N
        top_idx = np.argsort(pred)[::-1][:top_n]
        top_scores = pred[top_idx]

        return top_idx, top_scores   
        
    

In [39]:
model = MatrixFactorizationSGD(n_factors=50, learning_rate=0.0015613418566555307, epochs=90, 
                               reg='l2', reg_param= 0.3899716109944508, batch_size=124, n_users=n_users_global, n_items=n_movies_global)

model.fit(train_users, train_movies, train_ratings, train_timestamps)


In [40]:
test_pred = model.predict(test_users, test_movies, test_timestamps)
rmse = np.sqrt(mean_squared_error(test_ratings, test_pred))
print(rmse)

0.9966022607595695


In [41]:
# Pick any user (even one that only appears in the test set)
user_id = 1
top_item_indices, scores = model.get_top_n_recommendations(user_id, top_n=10)

recs = movies.iloc[top_item_indices][['movieId', 'title']].copy()
recs['predicted_rating'] = scores
recs.head(10)

,movieId,title,predicted_rating
7399,79895,"Extraordinary Adventures of Adèle Blanc-Sec, T...",4.532017
1104,1437,"Cement Garden, The (1993)",4.516363
285,327,Tank Girl (1995),4.487589
281,322,Swimming with Sharks (1995),4.487259
283,325,National Lampoon's Senior Trip (1995),4.487191
289,331,Tom & Viv (1994),4.487086
3066,4115,Hiding Out (1987),4.456612
2256,2993,Thunderball (1965),4.408701
2775,3713,"Long Walk Home, The (1990)",4.388363
2158,2874,"Pajama Game, The (1957)",4.383903
